# Lezione 05 — Distribuzioni di probabilità

**Corso:** Statistica Applicata con Python · **Blocco 2:** Probabilità
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/05_distribuciones_probabilidad_IT.md`

## Cosa vedrai in questa lezione

- **Variabili aleatorie**, funzione di massa di probabilità (pmf), densità (pdf), funzione di ripartizione (cdf) e quantili (ppf).
- Distribuzioni come **modelli dei dati**: binomiale, di Poisson, normale ed esponenziale, con `scipy.stats`.
- La **densità non è una probabilità**; lo **z-score** e la regola 68-95-99,7.
- La **t di Student**: la distribuzione di una *statistica*, non dei dati.
- Il **teorema del limite centrale** (TLC) con la simulazione, e un caso in cui fallisce (Cauchy).

## Prerequisiti

- Lezioni da 2 a 4 (z-score, ECDF e istogrammi, probabilità e simulazione).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy` (`pip install numpy pandas matplotlib scipy`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada; poi confrontalo con quello che vedi. Tutti i numeri degli esempi sono un *esempio illustrativo*, non dati reali.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** del generatore casuale: con lo stesso seme, tutti otteniamo gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. Variabile discreta: la binomiale

10 clienti visitano un negozio e ognuno acquista con probabilità 0,3 (*esempio illustrativo*). Sia X = numero di acquirenti. Con `stats.binom(n, p)` otteniamo la **pmf** (probabilità di ogni valore) e la **cdf** (probabilità cumulata).

In [ ]:
compras = stats.binom(n=10, p=0.3)
k = np.arange(0, 11)

tabla = pd.DataFrame({"k": k,
                      "pmf: P(X = k)": compras.pmf(k),
                      "cdf: P(X <= k)": compras.cdf(k)}).set_index("k")
display(tabla.round(4))
print("Somma della pmf:", compras.pmf(k).sum().round(6))
print("P(X >= 5) =", compras.sf(4).round(4), "  quantile 0,9:", compras.ppf(0.9))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(k, compras.pmf(k)); ax[0].set_title("Funzione di massa (pmf)")
ax[1].step(k, compras.cdf(k), where="post"); ax[1].set_title("Funzione di ripartizione (cdf): una scala")
for a in ax: a.set_xlabel("k")
plt.show()

**Esperimento guidato 1 — la simulazione coincide con la pmf?** Simuliamo 100 000 negozi con 10 clienti ciascuno e confrontiamo la frequenza relativa di ogni valore con la pmf teorica.

In [ ]:
sim = compras.rvs(size=100_000, random_state=rng)
frec = np.bincount(sim, minlength=11) / len(sim)

comp = pd.DataFrame({"teorica": compras.pmf(k), "simulata": frec}, index=k)
comp["differenza"] = (comp["simulata"] - comp["teorica"]).abs()
display(comp.round(4))

# tolleranza: 5 errori standard di una proporzione (caso peggiore p = 0,5)
tol = 5 * np.sqrt(0.25 / len(sim))
assert comp["differenza"].max() < tol
print("Media simulata:", sim.mean().round(3), "  teorica:", compras.mean(), "  varianza simulata:", sim.var().round(3), "  teorica:", round(compras.var(), 3))

**Cosa osservi?**

- Quanto sono vicine le frequenze simulate alla pmf? Dove è maggiore la differenza?
- La media e la varianza simulate coincidono con n·p e n·p·(1 − p)?
- Cosa accadrebbe alla forma della pmf se p fosse 0,05 invece di 0,3?

*(La tua risposta qui.)*

---
## 2. Conteggi: Poisson e sovradispersione

Un servizio di assistenza riceve in media 4 ticket all'ora (*esempio illustrativo*). La Poisson ha un tratto distintivo: **media = varianza**. Confrontiamo dati che rispettano questa uguaglianza con dati di conteggio con **sovradispersione** (stessa media, molta più varianza).

In [ ]:
tickets = stats.poisson(mu=4)
print("P(X = 6) =", tickets.pmf(6).round(4), "  P(X <= 2) =", tickets.cdf(2).round(4), "  P(X = 0) =", tickets.pmf(0).round(4))

n_horas = 20_000
datos_poisson = tickets.rvs(size=n_horas, random_state=rng)
# stessa media di 4 ticket, ma con ritmo variabile (binomiale negativa: varianza = 12)
datos_sobre = rng.negative_binomial(n=2, p=2 / 6, size=n_horas)

resumen = pd.DataFrame({
    "media": [datos_poisson.mean(), datos_sobre.mean()],
    "varianza": [datos_poisson.var(ddof=1), datos_sobre.var(ddof=1)],
}, index=["Poisson", "sovradispersione"])
resumen["var / media"] = resumen["varianza"] / resumen["media"]
display(resumen.round(3))

kk = np.arange(0, 21)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(kk - 0.2, np.bincount(datos_poisson, minlength=21)[:21] / n_horas, width=0.4, label="dati Poisson")
ax.bar(kk + 0.2, np.bincount(datos_sobre, minlength=21)[:21] / n_horas, width=0.4, label="dati sovradispersi")
ax.plot(kk, tickets.pmf(kk), "ko-", label="modello di Poisson (λ = 4)")
ax.set_xlabel("ticket all'ora"); ax.legend(); ax.set_title("Poisson a confronto con dati con sovradispersione")
plt.show()

print("P(X >= 12): modello di Poisson =", tickets.sf(11).round(5), " dati sovradispersi =", (datos_sobre >= 12).mean().round(5))

**Cosa osservi?**

- Qual è il rapporto varianza / media in ciascun insieme? Quale è compatibile con la Poisson?
- Quale probabilità di ≥ 12 ticket prevede il modello di Poisson e quanta ce n'è davvero nei dati sovradispersi?
- Quale conseguenza pratica avrebbe dimensionare il team con il modello sbagliato?

*(La tua risposta qui.)*

---
## 3. Variabile continua: densità, area e z-score

Altezze con media 170 cm e deviazione standard 7 cm (*esempio illustrativo*). Tre idee: (1) la **probabilità è un'area**, (2) la **densità può essere maggiore di 1**, (3) lo **z-score** traduce un valore in un percentile.

In [ ]:
altura = stats.norm(loc=170, scale=7)

# 1) probabilità = area = differenza della cdf
print("P(X < 180) =", altura.cdf(180).round(4), "  P(160 < X < 180) =", (altura.cdf(180) - altura.cdf(160)).round(4))
print("P(X = 170) = 0   ma la densità in 170 è", altura.pdf(170).round(4))

# 2) densità maggiore di 1 (e comunque l'area totale è 1)
estrecha = stats.norm(loc=0, scale=0.1)
xs = np.linspace(-1, 1, 20_001)
print("Densità massima della normale stretta:", estrecha.pdf(0).round(3),
      " area totale ≈", (estrecha.pdf(xs).sum() * (xs[1] - xs[0])).round(4))

# 3) z-score e percentile
z = (180 - 170) / 7
print("z =", round(z, 3), "  percentile:", (100 * stats.norm.cdf(z)).round(1), "%   97,5° percentile →", altura.ppf(0.975).round(2), "cm")

xs2 = np.linspace(145, 195, 500)
plt.plot(xs2, altura.pdf(xs2))
zona = (xs2 >= 160) & (xs2 <= 180)
plt.fill_between(xs2[zona], altura.pdf(xs2[zona]), alpha=0.4, label="P(160 < X < 180)")
plt.legend(); plt.xlabel("cm"); plt.ylabel("densità"); plt.title("La probabilità è l'area sotto la curva")
plt.show()

### La regola 68-95-99,7

Verifichiamo la regola con la cdf e con un campione simulato di 100 000 altezze.

In [ ]:
muestra = altura.rvs(size=100_000, random_state=rng)
filas = []
for kσ in (1, 2, 3):
    exacta = stats.norm.cdf(kσ) - stats.norm.cdf(-kσ)
    empirica = (np.abs(muestra - 170) < kσ * 7).mean()
    filas.append({"intervallo": f"μ ± {kσ}σ", "esatta": exacta, "simulata": empirica})
display(pd.DataFrame(filas).set_index("intervallo").round(4))

**Cosa osservi?** La proporzione esatta e quella simulata coincidono? Ora pensa agli stipendi della lezione 2 (molto asimmetrici): avrebbe senso dire che "il 95 % degli stipendi cade a meno di 2 deviazioni dalla media"?

*(La tua risposta qui.)*

---
## 4. Tempi di attesa: l'esponenziale

Il tempo fino alla chiamata successiva ha media 5 minuti (*esempio illustrativo*). È asimmetrica e **non ha memoria**.

In [ ]:
espera = stats.expon(scale=5)                      # scale = media = 1/λ
tiempos = espera.rvs(size=100_000, random_state=rng)

print("P(T > 10) =", espera.sf(10).round(4), "  simulata:", (tiempos > 10).mean().round(4))
print("mediana teorica:", espera.ppf(0.5).round(3), "  media teorica:", espera.mean())
print("Senza memoria: P(T > 15 | T > 5) =", (espera.sf(15) / espera.sf(5)).round(4), "  simulata:", (tiempos[tiempos > 5] > 15).mean().round(4))

plt.hist(tiempos, bins=60, density=True, alpha=0.5, label="simulazione")
xs = np.linspace(0, 30, 300)
plt.plot(xs, espera.pdf(xs), "k", label="pdf")
plt.axvline(espera.mean(), color="red", linestyle="--", label="media"); plt.axvline(espera.ppf(0.5), color="green", linestyle="--", label="mediana")
plt.legend(); plt.xlabel("minuti"); plt.title("Esponenziale: molto asimmetrica a destra")
plt.show()

**Cosa osservi?** Perché la mediana è minore della media? Cosa significa che P(T > 15 | T > 5) sia uguale a P(T > 10)?

*(La tua risposta qui.)*

---
## 5. La t di Student: la distribuzione di una statistica

**Attenzione:** la t **non** modella i dati. È la distribuzione della media standardizzata con la deviazione standard *stimata* s: T = (x̄ − μ) / (s / √n). Lo verifichiamo simulando 200 000 campioni di n = 5 dati da una normale (*esempio illustrativo*) e confrontando con la normale standard e con la t a 4 gradi di libertà.

In [ ]:
n = 5
datos = rng.normal(loc=0, scale=1, size=(200_000, n))
T = datos.mean(axis=1) / (datos.std(axis=1, ddof=1) / np.sqrt(n))

print("P(|T| > 2) simulata =", (np.abs(T) > 2).mean().round(4))
print("P(|T| > 2) t con 4 gl   =", (2 * stats.t.sf(2, df=4)).round(4))
print("P(|T| > 2) normale standard =", (2 * stats.norm.sf(2)).round(4))

filas = [{"gradi di libertà": gl, "P(|T| > 2)": 2 * stats.t.sf(2, df=gl)} for gl in (2, 4, 10, 30, 100)]
filas.append({"gradi di libertà": "normale", "P(|T| > 2)": 2 * stats.norm.sf(2)})
display(pd.DataFrame(filas).set_index("gradi di libertà").round(4))

xs = np.linspace(-6, 6, 500)
plt.hist(T[np.abs(T) < 6], bins=120, density=True, alpha=0.4, label="T simulata")
plt.plot(xs, stats.t.pdf(xs, df=4), label="t (4 gl)")
plt.plot(xs, stats.norm.pdf(xs), "k--", label="normale standard")
plt.legend(); plt.title("La t ha code più pesanti della normale")
plt.show()

**Cosa osservi?**

- La T simulata somiglia di più alla normale o alla t a 4 gradi di libertà? Dove si nota la differenza: al centro o nelle code?
- Cosa accade a P(|T| > 2) all'aumentare dei gradi di libertà?
- Ricorda: i dati erano normali e comunque T non è normale. Perché? (Suggerimento: cosa si stima con il campione?)

*(La tua risposta qui.)*

---
## 6. Esperimento guidato 2 — il teorema del limite centrale

Popolazione: **esponenziale con media 1**, molto asimmetrica (asimmetria teorica = 2). Per ogni n in {1, 5, 30, 200} ripetiamo 10 000 volte «prendere n dati e calcolarne la media». Teoria: la media delle medie è 1, la loro deviazione standard è 1/√n e la loro asimmetria è 2/√n.

Prima di eseguire, scrivi la tua previsione: che forma avrà l'istogramma delle medie con n = 200?

In [ ]:
NS = [1, 5, 30, 200]
REPS = 10_000
medias = {n: rng.exponential(1.0, size=(REPS, n)).mean(axis=1) for n in NS}

filas = []
for n in NS:
    m = medias[n]
    assert abs(m.mean() - 1) < 5 / np.sqrt(n * REPS)          # tolleranza: 5 errori standard
    filas.append({"n": n, "media delle medie": m.mean(), "dev. standard": m.std(),
                  "1/√n": 1 / np.sqrt(n), "asimmetria": stats.skew(m), "2/√n": 2 / np.sqrt(n)})
display(pd.DataFrame(filas).set_index("n").round(3))

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, n in zip(axes, NS):
    m = medias[n]
    ax.hist(m, bins=40, density=True, alpha=0.6)
    xs = np.linspace(m.min(), m.max(), 300)
    ax.plot(xs, stats.norm(1, 1 / np.sqrt(n)).pdf(xs), "k", lw=1.5)
    ax.set_title(f"n = {n}\nasimmetria = {stats.skew(m):.2f}")
plt.suptitle("Medie di n dati esponenziali, con la curva normale del TLC sovrapposta", y=1.03)
plt.show()

**Cosa osservi?**

- Che forma ha l'istogramma con n = 1? E con n = 200? A partire da quale n comincia ad assomigliare alla curva nera?
- Come cambia la larghezza (deviazione standard)? E l'asimmetria? Con n = 30 è già vicina a 0?
- Ogni singolo dato resta esponenziale. Cosa diventa approssimativamente normale: i dati o la media?

*(La tua risposta qui.)*

---
## 7. Quando il TLC fallisce: la distribuzione di Cauchy

La Cauchy ha forma a campana ma **non ha media né varianza finite** (code molto pesanti). Il TLC richiede varianza finita. Confrontiamo, man mano che si accumulano i dati, la **media cumulata** di un'esponenziale e quella di una Cauchy, e poi lo scarto interquartile delle medie con n = 10 e n = 1000.

In [ ]:
N = 5_000
exp_datos = rng.exponential(1.0, N)
cau_datos = rng.standard_cauchy(N)
pasos = np.arange(1, N + 1)

plt.plot(pasos, np.cumsum(exp_datos) / pasos, label="esponenziale (si stabilizza su 1)")
plt.plot(pasos, np.cumsum(cau_datos) / pasos, label="Cauchy (non si stabilizza)")
plt.axhline(1, color="black", linestyle="--")
plt.ylim(-6, 6); plt.xscale("log"); plt.legend(); plt.xlabel("numero di dati (scala log)"); plt.ylabel("media cumulata")
plt.title("La media di una Cauchy non converge")
plt.show()

REPS_C = 5_000
filas = []
for nombre, gen in (("esponenziale", lambda size: rng.exponential(1.0, size)), ("Cauchy", lambda size: rng.standard_cauchy(size))):
    iqr = {n: stats.iqr(gen((REPS_C, n)).mean(axis=1)) for n in (10, 1000)}
    filas.append({"distribuzione": nombre, "IQR (n = 10)": iqr[10], "IQR (n = 1000)": iqr[1000], "rapporto": iqr[1000] / iqr[10]})
display(pd.DataFrame(filas).set_index("distribuzione").round(3))

**Cosa osservi?**

- Cosa accade alla media cumulata dell'esponenziale e a quella della Cauchy aggiungendo dati? Ci sono "salti" nella Cauchy? Perché?
- Il rapporto tra scarti interquartili (n = 1000 rispetto a n = 10): quanto vale nei due casi? Cosa ti aspetteresti se il TLC funzionasse (1/√100 = 0,1)?
- Se i tuoi dati reali avessero ogni tanto valori estremi enormi, quale precauzione prenderesti prima di usare la normale per la media?

*(La tua risposta qui.)*

---
## Riepilogo della lezione

- Una **variabile aleatoria** assegna numeri agli esiti del caso; la **pmf** (discrete) dà P(X = x) e la **pdf** (continue) dà una densità: la probabilità è l'**area** e P(X = x) = 0.
- La **cdf** accumula probabilità; la **ppf** (quantili) è la sua inversa. In `scipy.stats`: `pmf`/`pdf`, `cdf`, `sf`, `ppf`, `rvs`.
- **Binomiale** (successi in n prove), **Poisson** (conteggi; media = varianza, altrimenti c'è sovradispersione), **normale** (campana; 68-95-99,7; z-score) ed **esponenziale** (attese) sono **modelli dei dati**.
- La **t di Student** è la distribuzione di una **statistica**, con code più pesanti e convergente alla normale.
- Il **TLC**: la **media** di n dati indipendenti con varianza finita è approssimativamente normale, con deviazione σ/√n; è un'approssimazione, con forte asimmetria n = 30 può non bastare e con la Cauchy non funziona.
- Prossima lezione: **stima e intervalli di confidenza** (lezione 6).